In [18]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [19]:
# states.py
from pydantic import BaseModel, Field
from typing import List

In [20]:
class GraphAgentState(BaseModel):
    question: str = Field(default="", description="User question about some math concept.")
    key_words: List[str] = Field(default_factory=list, description="Tags and concepts retrieved.")
    graph_context: str = Field(default="", description="Raw text retrieved from the Knowlodge Graph.")
    answer: str = Field(default="", description="Model final answer.")

In [45]:
# node_extractor.py
import json
import os 
from groq import Groq


GROQ_MODEL_NAME = "openai/gpt-oss-20b"
GROQ_EXTRACTION_TEMPERATURE = 0.1

def get_llm_client(provider_name: str = "groq", api_key: str = ""):
    if provider_name == "groq":
        return Groq(api_key=api_key)
    else: 
        return "Other client not yet implemented"
    
def llm_response(
    client, 
    provider_name: str = "", 
    system_prompt: str = "", 
    question: str = "", 
    model: str = "openai/gpt-oss-20b", 
    temperature: float = 0.1,
    json_mode: bool = False  # <-- Novo parâmetro
):
    if provider_name == "groq":
        # Preparamos os argumentos base
        kwargs = {
            "messages": [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": question}
            ],
            "temperature": temperature,
            "model": model
        }
        
        # Injetamos o JSON mode APENAS se for solicitado
        if json_mode:
            kwargs["response_format"] = {"type": "json_object"}

        # Fazemos a chamada desembrulhando os kwargs
        chat_completion = client.chat.completions.create(**kwargs)

        content = chat_completion.choices[0].message.content
        return content 
    else: 
        print("Provided model not implemented yet")
        return ""

In [46]:
SYSTEM_PROMPT = """You are a mathematical librarian. 
Analyze the user's question and extract 1 to 3 mathematical concepts or keywords. 
You MUST return ONLY a JSON object with the key "key_words" containing a list of strings (in English, to match the database). 
Example: {"key_words": ["polynomials", "roots", "unit circle"]}
"""

def retrieve_key_words(
    client, 
    system_prompt: str,
    state: GraphAgentState, 
    provider_name: str = "groq", 
    model: str = "openai/gpt-oss-20b", 
    temperature: float = 0.1
) -> dict:
    """Read user question and return key words""" 
    question = state.question
    print(f"Extracting key words from {question}")

    content = llm_response(
    client, 
    provider_name=provider_name, 
    system_prompt=system_prompt, 
    question=question, 
    model=model, 
    temperature=temperature, 
    json_mode=True
    )
    try:
        result_json = json.loads(content)

        extracted_tags = result_json.get("key_words", [])
        print(f"Extracted tags: {extracted_tags}")

        return {"key_words": extracted_tags}
    except Exception as error:
        print(f"Error: {error}")
        return {"key_words": []}

In [23]:
# fast test
from dotenv import load_dotenv
load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")

state = GraphAgentState(
    question="How can we calculate the chain rule for multidimension vectors?"
)

client = get_llm_client(api_key=GROQ_API_KEY)
key_words = retrieve_key_words(
    client, 
    SYSTEM_PROMPT, 
    state, 
    "groq" 
)

Extracting key words from How can we calculate the chain rule for multidimension vectors?
Extracted tags: ['chain rule', 'Jacobian matrix', 'partial derivatives']


In [24]:
# node_retriever.py
# from states import GraphAgentState
from math_assistant_agent.data import load_graph_json

GRAPH_DATA = load_graph_json("/home/cdrleo/Desktop/ai-projects/math-assistant-agent/data/graph_math_2026-07-17-16.json")

In [27]:
GRAPH_DATA["nodes"][0]

{'id': 'question_182412',
 'label': 'Question',
 'properties': {'question_id': 182412,
  'title': 'Why do roots of polynomials tend to have absolute value close to 1?',
  'text': 'While playing around with Mathematica I noticed that most polynomials with real coefficients seem to have most complex zeroes very near the unit circle. For instance, if we plot all the roots of a polynomial of degree 300 with coefficients chosen randomly from the interval $[27, 42]$, we get something like this:\n\nThe Mathematica code to produce the picture was:\n\nrandomPoly[n_, x_, {a_, b_}] := \n  x^Range[0, n] . Table[RandomReal[{a, b}], {n + 1}];\nGraphics[Point[{Re[x], Im[x]}] /. \n  NSolve[randomPoly[300, x, {27, 42}], x], Axes -> True]\n\nIf I try other intervals and other degrees, the picture is always mostly the same: almost all roots are close to the unit circle.\n\nQuestion: why does this happen?',
  'score': 446,
  'view_count': 69357,
  'link': 'https://mathoverflow.net/questions/182412/why-do-

In [30]:
labels = []
for node in GRAPH_DATA.get("nodes", []):
    labels.append(node["label"])
print(set(labels))

{'Tag', 'Question', 'Answer'}


In [31]:
GRAPH_DATA.get("edges", [])[0]

{'source': 'question_182412',
 'target': 'answer_182433',
 'type': 'HAS_ACCEPTED_ANSWER',
 'properties': {}}

In [33]:
edges = []
for edge in GRAPH_DATA.get("edges", []):
    edges.append(edge["type"])
print(set(edges))

{'TAGGED_WITH', 'HAS_ACCEPTED_ANSWER'}


In [28]:
GRAPH_DATA["edges"][0]

{'source': 'question_182412',
 'target': 'answer_182433',
 'type': 'HAS_ACCEPTED_ANSWER',
 'properties': {}}

In [ ]:
def retrieve_graph_context(
        state: GraphAgentState, 
        graph_data: dict
) -> dict:
    """Read key words from state, look for the information in the graph and return the context"""
    if isinstance(state, dict):
            key_words = state.get("key_words", [])
    else:
        key_words = state.key_words

    print(f"Looking for tags like {key_words}")

    if not key_words:
        print(f"'key_words' is empty")
        return {"graph_context": ""}
    
    contexts = []

    for node in graph_data.get("nodes", []):
        if node["label"] == "Question":
            question = node["properties"].get("text", "").lower()
            title = node["properties"].get("title", "").lower()

            match_ = False
            for kw in key_words:
                if kw.lower() in question or kw.lower() in title:
                    match_ = True
                    break
            
            if match_:
                question_id = node["id"]
                context = ""
                answer_id = None

                for edge in graph_data.get("edges", []):
                    if edge["source"] == question_id and edge["type"] == "HAS_ACCEPTED_ANSWER":
                        answer_id = edge["target"]
                        break 
                
                if answer_id:
                    for answer_node in graph_data.get("nodes", []):
                        if answer_node["id"] == answer_id:
                            context = answer_node["properties"].get("text", "")
                            break

                context_block = f"""
                RELATED PROBLEM: 
                    title: {node["properties"].get("title", "")}
                    accepted solution: {context}
                """

                contexts.append(context_block)
    
    full_context = "\n\n".join(contexts)

    if not full_context:
        full_context = "None information found."
        print(f"KG does not have information for the selected key words: {key_words}")
    else:
        print(f"Some informations founded.")
    
    return {"graph_context": full_context}

In [35]:
type(GRAPH_DATA)

dict

In [36]:
# fast test
# from states import GraphAgentState
# from node_retriever import retrieve_graph_context

MOCK_STATE = GraphAgentState(
    question="Why do roots of polynomials fall on the unit circle?",
    key_words=["polynomials", "unit circle"]
)

result = retrieve_graph_context(state=MOCK_STATE, graph_data=GRAPH_DATA)

print(result["graph_context"])

Looking for tags like ['polynomials', 'unit circle']
Some informations founded.

                RELATED PROBLEM: 
                    title: Why do roots of polynomials tend to have absolute value close to 1?
                    accepted solution: Let me give an informal explanation using what little I know about complex analysis.

Suppose that $p(z)=a_{0}+\dotsm+a_{n}z^{n}$ is a polynomial with random complex coefficients and suppose that $p(z)=a_{n}(z-c_{1})\cdots(z-c_{n})$. Then take note that

$$\frac{p'(z)}{p(z)}=\frac{d}{dz}\log(p(z))=\frac{d}{dz}\log(z-c_{1})+\dotsm+\log(z-c_{n})=
\frac{1}{z-c_{1}}+\dotsm+\frac{1}{z-c_{n}}.
$$

Now assume that $\gamma$ is a circle larger than the unit circle. Then

$$\oint_{\gamma}\frac{p'(z)}{p(z)}dz=\oint_{\gamma}\frac{na_{n}z^{n-1}+(n-1)a_{n-1}z^{n-2}+\dotsm+a_{1}}{a_{n}z^{n}+\dotsm+a_{0}}\approx\oint_{\gamma}\frac{n}{z}dz=2\pi in.$$

However, by the residue theorem,

$$\oint_{\gamma}\frac{p'(z)}{p(z)}dz=\oint_{\gamma}\frac{1}{z-c_{1}}+...+\

In [37]:
# node_generator.py
import os 
from groq import Groq
# from states import GraphAgentState
# from node_extractor import get_llm_client

In [47]:
def brain(
    client, 
    state: GraphAgentState, 
    provider_name: str = "groq", 
    model: str = "openai/gpt-oss-20b", 
    temperature: float = 0.2, 
):
    """Read the user question and the retrieved context and generate the final answer"""
    if isinstance(state, dict):
        question = state.get("question", "")
        graph_context = state.get("graph_context", "")
    else:
        question = state.question
        graph_context = state.graph_context 

    print(f"Generating the final answer for question {question[:30]}... (print truncated).")

    # Checagem de segurança se o Nó 2 não encontrou nada
    if not graph_context or "None information" in graph_context:
        answer = "I'm sorry, I couldn't find relevant information about this question in the Knowledge Graph."
        print("KG context is empty.")
        return {"answer": answer}
    
    system_prompt = f"""
    You are an expert Math Teaching Assistant.
    Your goal is to answer the user's question clearly, using ONLY the context provided below.
    If the context contains LaTeX math, format your response using proper LaTeX delimiters ($ for inline, $$ for block).
    Break your explanation into logical steps. Do not hallucinate math theorems outside the context.

    CONTEXT RETRIEVED FROM KNOWLEDGE GRAPH:
    {graph_context}
    """

    try:
        # ATENÇÃO: Garanta que a sua função llm_response não esteja forçando 
        # JSON mode aqui, ou o LLM vai quebrar!
        result_text = llm_response(
            client=client,
            provider_name=provider_name, 
            system_prompt=system_prompt, 
            question=question, 
            model=model, 
            temperature=temperature, 
            json_mode=False
        )

        print("Answer generated successfully.")

        # CORREÇÃO 1: Retornando a variável correta gerada pelo LLM
        return {"answer": result_text}
    
    except Exception as error:
        print(f"Error generating answer: {error}")
        # CORREÇÃO 2: Retornando um dicionário válido para o LangGraph não quebrar
        return {"answer": "An error occurred while generating the mathematical response."}

In [48]:
# fast test
# from states import GraphAgentState
# from node_generator import brain

GRAPH_CONTEXT = result["graph_context"]
state = GraphAgentState(
    question="Why do roots of polynomials fall on the unit circle?",
    key_words=["polynomials", "unit circle"],
    graph_context=GRAPH_CONTEXT
)

brain_result = brain(
    client=client, 
    state=state
)

print(brain_result["answer"])

Generating the final answer for question Why do roots of polynomials fa... (print truncated).
Answer generated successfully.
**Why do the zeros of a “generic’’ polynomial tend to lie on the unit circle?**  
Below is a concise, step‑by‑step outline of the informal argument that appears in the cited solution.

---

### 1.  Write the polynomial in factored form  

Let  

\[
p(z)=a_{0}+a_{1}z+\dots +a_{n}z^{n}
      =a_{n}(z-c_{1})(z-c_{2})\cdots(z-c_{n}),
\]

where \(c_{1},\dots ,c_{n}\) are the (complex) zeros of \(p\).

---

### 2.  Use the logarithmic derivative  

\[
\frac{p'(z)}{p(z)}
   =\frac{d}{dz}\log p(z)
   =\frac{1}{z-c_{1}}+\frac{1}{z-c_{2}}+\dots +\frac{1}{z-c_{n}} .
\]

The poles of this meromorphic function are exactly the zeros of \(p\).

---

### 3.  Integrate over a large circle  

Let \(\gamma\) be the circle \(|z|=R\) with \(R>1\).  
For \(|z|=R\) the dominant term in the numerator of \(p'(z)/p(z)\) is \(na_{n}z^{\,n-1}\), while the dominant term in the denominator is

In [49]:
# graph_flow.py
import os 
from langgraph.graph import StateGraph, START, END 
from IPython.display import Image, display

# from states import GraphAgentState
# from node_extractor import retrieve_key_words, get_llm_client, SYSTEM_PROMPT
# from node_retriever import retrieve_graph_context
# from node_generator import brain
from math_assistant_agent.data import load_graph_json

# ==========================================
# Configs and Dependencies
# ==========================================
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
client = get_llm_client(api_key=GROQ_API_KEY)

GRAPH_DATA = load_graph_json("/home/cdrleo/Desktop/ai-projects/math-assistant-agent/data/graph_math_2026-07-17-16.json")

In [53]:
# ==========================================
# Wrappers (Langgraph only uses state)
# ==========================================

def wrapper_extract_keywords(state: GraphAgentState):
    return retrieve_key_words(
        client=client, 
        system_prompt=SYSTEM_PROMPT, 
        state=state, 
        provider_name="groq", 
        model="openai/gpt-oss-20b", 
    )

def wrapper_retrieve_context(state: GraphAgentState):
    return retrieve_graph_context(
        state=state, graph_data=GRAPH_DATA, 
    )

def wrapper_brain(state: GraphAgentState):
    return brain(
        client=client, state=state, provider_name="groq", model="openai/gpt-oss-20b"
    )

# ==========================================
# Graph Building (StateGraph)
# ==========================================


workflow = StateGraph(GraphAgentState)

workflow.add_node("extractor", wrapper_extract_keywords)
workflow.add_node("retriever", wrapper_retrieve_context)
workflow.add_node("brain", wrapper_brain)

workflow.add_edge(START, "extractor")
workflow.add_edge("extractor", "retriever")
workflow.add_edge("retriever", "brain")
workflow.add_edge("brain", END)

agent = workflow.compile()

In [54]:
try:
    image_data = agent.get_graph().draw_mermaid_png()
    with open("outputs/agent_architecture.png", "wb") as f:
        f.write(image_data)
    print("📸 output path 'outputs/agent_architecture.png'!")
except Exception as e:
    print(f"Error: {e}")

📸 output path 'outputs/agent_architecture.png'!


In [55]:
# fast test
if __name__ == "__main__":
    print("\n🚀 STARTING AGENT EXECUTION...")
    
    QUESTION = "Can you explain how we can calculate the integral of x^2?"
    
    inputs = GraphAgentState(question=QUESTION)
    
    ANSWER = agent.invoke(inputs)
    
    print("\n" + "="*50)
    print("🎓 AGENT ANSWER:")
    print("="*50)
    print(ANSWER["answer"])


🚀 STARTING AGENT EXECUTION...
Extracting key words from Can you explain how we can calculate the integral of x^2?
Extracted tags: ['integration', 'polynomial', 'antiderivative']
Looking for tags like ['integration', 'polynomial', 'antiderivative']
Some informations founded.
Generating the final answer for question Can you explain how we can cal... (print truncated).
Answer generated successfully.

🎓 AGENT ANSWER:
To find the antiderivative (indefinite integral) of the function \(x^{2}\), we use the **power rule for integration**.  
The rule states that for any real exponent \(n\neq-1\),

\[
\int x^{\,n}\,dx=\frac{x^{\,n+1}}{n+1}+C,
\]

where \(C\) is the constant of integration.

---

### Step 1: Identify the exponent  
Here the integrand is \(x^{2}\), so \(n=2\).

### Step 2: Apply the power rule  
Insert \(n=2\) into the formula:

\[
\int x^{2}\,dx=\frac{x^{\,2+1}}{2+1}+C=\frac{x^{3}}{3}+C.
\]

### Step 3: Write the result  
Thus the antiderivative of \(x^{2}\) is

\[
\boxed{\displa